This is part 14 of a tutorial series. We recommend following them in order, starting with [Part 0: Welcome to `musica`](0.%20Welcome%20to%20MUSICA.ipynb).

# Introduction to DAE Solving for Atmospheric Chemistry

In atmospheric chemistry models, some processes — like gas-liquid partitioning and aqueous dissociation equilibria — are orders of magnitude faster than the gas-phase kinetics we're integrating. As modelers, we're used to handling these fast processes separately from the ODE integration. This tutorial explains why that approach introduces artifacts and how **Differential-Algebraic Equation (DAE) solvers** eliminate them.

## 1. The Problem: Process Splitting

Consider a gas-phase species $A_g$ that dissolves into cloud droplets as $A_{aq}$, where it is irreversibly consumed by an aqueous-phase reaction:

$$A_g \underset{\text{Henry's Law}}{\rightleftharpoons} A_{aq} \xrightarrow{k} \text{products}$$

The partitioning timescale is typically microseconds, while the aqueous kinetics and gas-phase evolution happen on timescales of seconds to minutes. The ODE solver's timestep is set by the slow processes, so the fast partitioning can't be resolved by the ODE.

**The typical workaround** is **operator splitting** (also called **process splitting**):

1. Advance the gas-phase ODE by one timestep $\Delta t$
2. Compute Henry's Law equilibrium analytically: $[A_{aq}] = K_H \cdot R T \cdot [A_g]$
3. Advance the aqueous kinetics for $\Delta t$
4. Re-equilibrate gas/aqueous via Henry's Law

This works, but it introduces **splitting errors**:
- The gas and aqueous chemistry are **decoupled** within each timestep — the aqueous loss of $A_{aq}$ doesn't feed back into $[A_g]$ until the next step
- Results depend on the **order** of the splitting steps
- The error is proportional to $\Delta t$ — you need very small timesteps to converge

Let's see this in practice.

## 2. Importing Libraries

In [ ]:
import math
import musica
import musica.mechanism_configuration as mc
from musica.micm import MICM, SolverType, SolverState
from musica.mechanism_configuration import (
    HenryLawConstant,
    UniformSection,
    DissolvedReaction,
    HenryLawEquilibrium,
    LinearConstraint,
    LinearConstraintTerm,
    DiagnoseFromState,
    Aerosol,
)
import matplotlib.pyplot as plt
import numpy as np

## 3. A Minimal System

We'll use the simplest possible system to isolate the splitting artifact:

- **Gas species**: $A_g$ (e.g., SO₂)
- **Aqueous species**: $A_{aq}$ (dissolved in cloud water)
- **Solvent**: H₂O
- **Henry's Law**: $[A_{aq}] = \alpha \cdot [A_g]$ where $\alpha = K_H \cdot R \cdot T \cdot f_v$ and $f_v$ is the liquid water volume fraction
- **Aqueous loss**: $A_{aq} \xrightarrow{k} \text{products}$ with rate constant $k$
- **Mass conservation**: $[A_g] + [A_{aq}] = $ total $A$

Let's define the constants. We'll use a Henry's Law constant chosen so that — with a realistic cloud liquid water content — the gas and aqueous pools are comparable in size. This makes the splitting error visible.

In [ ]:
# Physical constants
R_GAS = 8.314          # J/(mol·K)
T = 280.0              # K
P = 70000.0            # Pa
MW_H2O = 0.018         # kg/mol
RHO_H2O = 1000.0       # kg/m³

# Cloud liquid water content
# In MIAM, the solvent concentration is "moles of liquid water in all
# cloud droplets per cubic meter of air" — NOT the molarity of pure water.
# A typical cloud has LWC ≈ 0.3 g/m³ of air.
LWC = 0.3e-3           # kg/m³ of air (= 0.3 g/m³)
C_H2O = LWC / MW_H2O   # mol/m³ of air ≈ 0.017
f_v = C_H2O * MW_H2O / RHO_H2O  # liquid volume fraction (m³ liquid / m³ air)
print(f"Cloud LWC = {LWC*1e3:.1f} g/m³  →  C_H2O = {C_H2O:.4f} mol/m³  →  f_v = {f_v:.2e}")

# Henry's Law constant for our test species
# MIAM's Henry's Law constraint computes:
#   [A_aq] = K_H * R * T * f_v * [A_g]
# where f_v = [H2O] * Mw / ρ is the liquid water volume fraction.
# We choose K_H so the effective alpha ≈ 1 at our cloud conditions.
HLC_REF = 1e3          # mol/(m³·Pa) at 298.15 K (large — compensates for tiny f_v)
HLC_C = 2000.0         # K (temperature dependence)
T0 = 298.15

# Compute effective Henry's Law partitioning ratio
HLC_T = HLC_REF * math.exp(HLC_C * (1.0/T - 1.0/T0))
alpha = HLC_T * R_GAS * T * f_v 
print(f"Henry's Law alpha = {alpha:.3f} (aq/gas concentration ratio)")
print(f"Fraction in gas phase: {1/(1+alpha):.3f}")
print(f"Fraction in aqueous phase: {alpha/(1+alpha):.3f}")

# Aqueous loss rate constant
# Large enough that significant loss happens per splitting step
k_loss = 0.05  # s⁻¹

# Initial condition
A_total_0 = 1e-6  # mol/m³ total A

## 4. Approach 1: Process Splitting (the traditional way)

Here we'll manually implement operator splitting:

1. Start with $[A_g]$ and $[A_{aq}]$ in Henry's Law equilibrium
2. Each timestep:
   - Apply aqueous loss: $[A_{aq}]' = [A_{aq}] \cdot e^{-k \Delta t}$
   - Re-equilibrate: redistribute total remaining $A$ between gas and aqueous phases

This is conceptually what most atmospheric models do (often with more sophisticated splitting methods, but the same fundamental limitation).

>**NOTE:** This code is intentionally showing the "traditional" approach to motivate the DAE alternative. It's not how you should do this in `musica`!

In [ ]:
def process_splitting(total_time, dt):
    """Simulate A_g <-> A_aq -> products using operator splitting."""
    # Start at Henry's Law equilibrium
    A_g = A_total_0 / (1 + alpha)
    A_aq = alpha * A_g
    
    times = [0.0]
    A_g_hist = [A_g]
    A_aq_hist = [A_aq]
    
    t = 0.0
    while t < total_time - 1e-10:
        step = min(dt, total_time - t)
        
        # Step 1: Apply aqueous-phase loss (decoupled from gas phase!)
        A_aq_after_loss = A_aq * math.exp(-k_loss * step)
        
        # Step 2: Re-equilibrate via Henry's Law
        A_remaining = A_g + A_aq_after_loss
        A_g = A_remaining / (1 + alpha)
        A_aq = alpha * A_g
        
        t += step
        times.append(t)
        A_g_hist.append(A_g)
        A_aq_hist.append(A_aq)
    
    return np.array(times), np.array(A_g_hist), np.array(A_aq_hist)

Let's run the splitting approach at several different timesteps to see how the results depend on $\Delta t$:

In [ ]:
total_time = 200.0  # s
dt_values = [50.0, 20.0, 5.0, 1.0]

split_results = {}
for dt in dt_values:
    t, A_g, A_aq = process_splitting(total_time, dt)
    split_results[dt] = (t, A_g, A_aq)

## 5. Approach 2: DAE Solving with MIAM

Now let's solve the same system the right way: encoding Henry's Law as an **algebraic constraint** in a DAE system. Instead of splitting, the solver enforces $[A_{aq}] = \alpha \cdot [A_g]$ **continuously** while integrating the aqueous kinetics.

This is exactly what MIAM does. Let's set it up:

- A `HenryLawEquilibrium` constraint replaces the manual equilibration
- A `DissolvedReaction` handles the aqueous loss
- A `LinearConstraint` enforces mass conservation
- The DAE solver (Rosenbrock with mass matrix) handles everything simultaneously

In [ ]:
# ── Species and phases ──
A_gas_sp = mc.Species(name="A_g")
gas = mc.Phase(name="gas", species=[A_gas_sp])

A_aq_sp = mc.Species(name="A_aq")
product_sp = mc.Species(name="Product")
h2o_sp = mc.Species(name="H2O")
h2o_sp.molecular_weight_kg_mol = MW_H2O
h2o_sp.density_kg_m3 = RHO_H2O

aq_phase = mc.Phase(
    name="AQUEOUS",
    species=[h2o_sp, A_aq_sp, product_sp],
)

# Cloud droplet representation
cloud = UniformSection(
    name="CLOUD",
    phases=[aq_phase],
    min_radius=1e-6,
    max_radius=1e-5,
)

# Aqueous-phase loss reaction: A_aq -> Product
# This is first-order (1 reactant), so the DissolvedReaction rate formula is:
#   rate = k / [solvent]^(n_reactants - 1) * [A_aq]
#        = k / [solvent]^0 * [A_aq]
#        = k * [A_aq]
# No C_H2O_M correction needed for unimolecular reactions.
# (The C_H2O_M factor is only needed for bimolecular reactions to cancel the 1/[solvent] denominator.)
# Rate constants are keyed by aerosol representation; a plain Python callable f(T) works too.
loss_rxn = DissolvedReaction(
    phase=aq_phase,
    reactants=[A_aq_sp],
    products=[product_sp],
    solvent=h2o_sp,
    rate_constants={cloud: lambda T: k_loss},
)

# Constraint 1: Henry's Law equilibrium
hlc_constraint = HenryLawEquilibrium(
    gas_phase=gas,
    gas_species=A_gas_sp,
    condensed_phase=aq_phase,
    condensed_species=A_aq_sp,
    solvent=h2o_sp,
    henry_law_constant=HenryLawConstant(HLC_ref=HLC_REF, C=HLC_C),
    solvent_molecular_weight=MW_H2O,
    solvent_density=RHO_H2O,
)

# Constraint 2: Mass conservation  A_g + A_aq + Product = total
# Using DiagnoseFromState lets the solver compute the total from the
# current state at the start of each solve step, which is essential for
# real-world use where the total may change between steps
mass_constraint = LinearConstraint(
    algebraic_phase=gas,
    algebraic_species=A_gas_sp,
    terms=[
        LinearConstraintTerm(gas, A_gas_sp, 1.0),
        LinearConstraintTerm(aq_phase, A_aq_sp, 1.0),
        LinearConstraintTerm(aq_phase, product_sp, 1.0),
    ],
    constant=DiagnoseFromState(),
)

# One Mechanism carries the gas phase AND the aerosol section;
# MIAM realizes the aerosol section at solve time.
mechanism = mc.Mechanism(
    name="dae_demo",
    species=[A_gas_sp, A_aq_sp, product_sp, h2o_sp],
    phases=[gas, aq_phase],
    reactions=[],
    aerosol=Aerosol(
        representations=[cloud],
        processes=[loss_rxn],
        constraints=[hlc_constraint, mass_constraint],
    ),
)

print("Mechanism with aerosol section created successfully!")

Now let's create the DAE solver and run it at different timesteps. The key difference: we use `SolverType.rosenbrock_dae4_standard_order` instead of the standard Rosenbrock solver. The "DAE" variants add mass-matrix support, which is how the algebraic constraints are enforced.

$M \frac{dy}{dt} = f(y)$

where $M_{ii} = 1$ for differential (ODE) variables and $M_{ii} = 0$ for algebraic (constraint) variables.

In [ ]:
def dae_solve(total_time, dt):
    """Simulate the same system using the DAE solver."""
    micm = MICM(
        mechanism=mechanism,
        solver_type=SolverType.rosenbrock_dae4_standard_order,
        external_models=[musica.MIAM()],
    )
    state = micm.create_state()
    mechanism.aerosol.set_default_parameters(state)
    state.set_conditions(temperatures=T, pressures=P)

    # Set initial conditions at Henry's Law equilibrium
    A_g_0 = A_total_0 / (1 + alpha)
    A_aq_0 = alpha * A_g_0
    state.set_concentrations({
        "A_g": A_g_0,
        "CLOUD.AQUEOUS.A_aq": A_aq_0,
        "CLOUD.AQUEOUS.Product": 0.0,
        "CLOUD.AQUEOUS.H2O": C_H2O,
    })

    times = [0.0]
    A_g_hist = [A_g_0]
    A_aq_hist = [A_aq_0]

    t = 0.0
    while t < total_time - 1e-10:
        step = min(dt, total_time - t)
        result = micm.solve(state, time_step=step)
        assert result.state == SolverState.Converged, f"Solver failed at t={t:.2f}s"
        t += step

        concs = state.get_concentrations()
        times.append(t)
        A_g_hist.append(concs["A_g"][0])
        A_aq_hist.append(concs["CLOUD.AQUEOUS.A_aq"][0])

    return np.array(times), np.array(A_g_hist), np.array(A_aq_hist)

dae_results = {}
for dt in dt_values:
    t, A_g, A_aq = dae_solve(total_time, dt)
    dae_results[dt] = (t, A_g, A_aq)

print("DAE solves complete.")

## 6. Comparing the Results

Now let's plot both approaches side-by-side. We'll focus on $[A_g]$ since it's the quantity a host model feeds to other processes. The splitting error shows up as a difference between the coarse and fine timestep solutions — for the DAE solver, there should be almost no difference.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(dt_values)))

# Left panel: Process splitting
ax = axes[0]
for (dt, color) in zip(dt_values, colors):
    t, A_g, A_aq = split_results[dt]
    ax.plot(t, A_g * 1e6, '-o', color=color, markersize=3,
            label=f'Δt = {dt:.0f} s')
ax.set_xlabel('Time (s)')
ax.set_ylabel('[A_g] (μmol/m³)')
ax.set_title('Process Splitting (traditional)')
ax.legend()
ax.grid(True, alpha=0.3)

# Right panel: DAE solver
ax = axes[1]
for (dt, color) in zip(dt_values, colors):
    t, A_g, A_aq = dae_results[dt]
    ax.plot(t, A_g * 1e6, '-o', color=color, markersize=3,
            label=f'Δt = {dt:.0f} s')
ax.set_xlabel('Time (s)')
ax.set_title('DAE Solver (MIAM)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Notice how:
- **Process splitting** (left): The coarse-timestep solutions diverge from the fine-timestep reference. At $\Delta t = 50$ s, the gas-phase concentration decays too slowly — because the aqueous loss and gas resupply are decoupled within each step.
- **DAE solver** (right): All timesteps produce essentially the same result. The algebraic constraint keeps the gas-aqueous coupling exact within each solver stage, regardless of $\Delta t$.

Let's quantify the error:

In [ ]:
# Use the initial gas-phase concentration for normalization
# (avoids division by near-zero when final concentrations are very small)
t_ref, A_g_ref, _ = split_results[1.0]
A_g_0 = A_g_ref[0]

print(f"{'Δt (s)':<10} {'Split error (%)':<20} {'DAE error (%)':<20}")
print('-' * 50)

for dt in dt_values[:-1]:  # compare coarser timesteps to Δt=1s reference
    # Splitting: compare final A_g to finest splitting result
    _, A_g_split, _ = split_results[dt]
    split_err = abs(A_g_split[-1] - A_g_ref[-1]) / A_g_0 * 100

    # DAE: compare final A_g to finest DAE result
    _, A_g_dae, _ = dae_results[dt]
    _, A_g_dae_ref, _ = dae_results[1.0]
    dae_err = abs(A_g_dae[-1] - A_g_dae_ref[-1]) / A_g_0 * 100

    print(f"{dt:<10.0f} {split_err:<20.4f} {dae_err:<20.6f}")

## 7. How it Works: The Mass Matrix

Under the hood, the DAE solver uses a **mass matrix** $M$ to distinguish between differential and algebraic variables. The system being solved is:

$$M \frac{dy}{dt} = f(y)$$

where:
- For **differential variables** (species governed by ODEs, like `Product`): $M_{ii} = 1$, so the equation is the normal ODE $\frac{dy_i}{dt} = f_i(y)$
- For **algebraic variables** (species governed by constraints, like `A_aq`, `A_g`): $M_{ii} = 0$, so the equation becomes $0 = f_i(y)$, which is a constraint equation

If you think about it, this is exactly analogous to what your steady-state calculation does — you're setting $\frac{d[A_{aq}]}{dt} = 0$ and solving. The difference is that the DAE solver does this **simultaneously** with the kinetics, so the feedback between fast and slow processes is fully captured.

MIAM handles all the mass-matrix construction for you. You just declare your constraints, and the solver takes care of the rest.

## 8. Available DAE Solver Types

MUSICA provides several DAE-capable solver variants:

| Solver Type | Description |
|---|---|
| `SolverType.rosenbrock_dae4_standard_order` | 4-stage DAE Rosenbrock (recommended for most cases) |
| `SolverType.rosenbrock_dae4` | 4-stage DAE Rosenbrock (optimized memory order) |
| `SolverType.rosenbrock_dae6_standard_order` | 6-stage DAE Rosenbrock (higher accuracy, more expensive) |
| `SolverType.rosenbrock_dae6` | 6-stage DAE Rosenbrock (optimized memory order) |

**When to use which:**
- `rosenbrock_dae4_standard_order` is the best default for Python-based applications
- `rosenbrock_dae6` variants give higher accuracy for very stiff systems with many constraints
- The non-`standard_order` variants use optimized memory layouts for HPC applications (columns, 3D grids)

**When do you need a DAE solver?** A good rule of thumb: if a process timescale is much shorter than your solver timestep (say, 100× shorter), constraining it will be both more accurate and more efficient than trying to resolve it with the ODE.

## 9. Summary

| | Process Splitting | DAE Solver |
|---|---|---|
| Gas-aqueous coupling | Decoupled within each timestep | Coupled throughout |
| Timestep dependence | Error ~ $\Delta t$ | Converged at all timesteps |
| Order dependence | Results depend on splitting order | No splitting; single coupled system |
| Implementation | Manual equilibration steps | Declare constraints; solver handles the rest |

In the [next tutorial](15.%20miam_processes_and_constraints.ipynb), we'll go through all the MIAM building blocks — processes, constraints, and representations — so you can set up your own coupled gas-aqueous systems.